In [ ]:
import requests
import json
import time
import yaml

# Define API URL and model name
OLLAMA_API_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "llama3.1:8b"

# Read the prompt from a text file
with open("prompt.txt", "r", encoding="utf-8") as file:
    base_prompt = file.read().strip()  # Remove extra spaces or newlines

# Read classification inputs from 'inputs.yml'
with open("User_curpus_queries.yml", "r", encoding="utf-8") as file:
    inputs_data = yaml.safe_load(file)  # Load YAML data
    classification_inputs = inputs_data.get("classification_inputs", [])  # Extract inputs

# Combine the base prompt with user inputs
full_prompt = base_prompt + "\n\nNow classify the following inputs:\n" + "\n".join(classification_inputs)
#print(full_prompt)
# Define conversation format
messages = [
    {"role": "system", "content": "You are a critical thinker data annotator. Please label the user inputs correctly by referring to the few-shot examples."},
    {"role": "user", "content": full_prompt}
]

# Start execution timer
start_time = time.time()

# Send request to Ollama
response = requests.post(
    OLLAMA_API_URL, 
    json={"model": MODEL_NAME, "messages": messages},
    stream=True
)

# Check if request was successful
if response.status_code != 200:
    print(f"Error: Received status code {response.status_code}")
    print(response.text)  # Print the error message
    exit()

# Process response line by line
print("Classification Results:\n")
for line in response.iter_lines(decode_unicode=True):
    if line.strip():
        try:
            data = json.loads(line)  # Decode JSON line-by-line
            content = data.get("message", {}).get("content", "")  # Extract content
            print(content, end="", flush=True)  # Display streaming output
        except json.JSONDecodeError as e:
            print(f"\nJSON Decode Error: {e}")  # Handle errors

# End execution timer
end_time = time.time()
print("\n")  # Corrected newline print
print(f"Execution time: {end_time - start_time:.2f} seconds")  # Print execution time


In [12]:
import requests
import json
import time
import yaml
import csv

# Define API URL and model name
OLLAMA_API_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "llama3.1:8b"

# Read the prompt from a text file
with open("prompt.txt", "r", encoding="utf-8") as file:
    base_prompt = file.read().strip()  # Remove extra spaces or newlines

# Read classification inputs from 'User_curpus_queries.yml'
with open("User_curpus_queries.yml", "r", encoding="utf-8") as file:
    inputs_data = yaml.safe_load(file)  # Load YAML data
    classification_inputs = inputs_data.get("classification_inputs", [])  # Extract inputs

# Combine the base prompt with user inputs
full_prompt = base_prompt + "\n\nNow classify the following inputs:\n" + "\n".join(classification_inputs)

# Define conversation format
messages = [
    {"role": "system", "content": "You are a critical thinker data annotator. Please label the user inputs correctly by referring to the few-shot examples."},
    {"role": "user", "content": full_prompt}
]

# Start execution timer
start_time = time.time()

# Send request to Ollama
response = requests.post(
    OLLAMA_API_URL, 
    json={"model": MODEL_NAME, "messages": messages},
    stream=True
)

# Check if request was successful
if response.status_code != 200:
    print(f"Error: Received status code {response.status_code}")
    print(response.text)  # Print the error message
    exit()

# Open a CSV file to save results
csv_file_path = "classification_results.csv"
results = []  # Store the results before writing to CSV

# Read the full response at once
full_response = ""

for line in response.iter_lines(decode_unicode=True):
    if line.strip():
        try:
            data = json.loads(line)  # Decode JSON line-by-line
            content = data.get("message", {}).get("content", "")  # Extract content
            full_response += content  # Append full response
        except json.JSONDecodeError as e:
            print(f"\nJSON Decode Error: {e}")  # Handle errors

# Split response into individual classifications
classified_lines = full_response.split("\n")

# Match inputs to their classifications
for i, classified_text in enumerate(classified_lines):
    if "→" in classified_text:  # Ensure we are parsing the correct format
        user_query, predicted_label = classified_text.split("→")
        results.append([user_query.strip().strip('"'), predicted_label.strip()])

# Save results to CSV
with open(csv_file_path, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["User Query", "Predicted Label"])  # CSV Header
    writer.writerows(results)  # Write all results

# End execution timer
end_time = time.time()
print("\n")  # Corrected newline print
print(f"Execution time: {end_time - start_time:.2f} seconds")  # Print execution time
print(f"\n✅ Classification results saved in: {csv_file_path}")




Execution time: 23.45 seconds

✅ Classification results saved in: classification_results.csv
